## Search from Item Inventory

running qdrant locally/docker

```docker run -p 6333:6333 -p 6334:6334 -v qdrant_data:/qdrant/storage qdrant/qdrant```


In [2]:
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import numpy as np
from qdrant_client.models import Filter, FieldCondition, MatchValue
import uuid

# --- Step 1: Initialize model and Qdrant client ---
model = SentenceTransformer('all-MiniLM-L6-v2')
qdrant = QdrantClient(":memory:")  # or "http://localhost:6333" for persistent server

collection_name = "closet_vectors"
dimension = 384  # dimension

# --- Step 2: Create collection ---
qdrant.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=dimension, distance=Distance.COSINE)
)

# --- Step 3: Closet Items and Embedding ---
closet_items = [
    # FORMAL / PROFESSIONAL
    {
        "id": "1",
        "name": "Black Wrap Dress",
        "description": "A black wrap dress suitable for formal events.",
        "category": "dress",
        "style": "formal",
        "season": "Autumn",
        "color": "black"
    },
    {
        "id": "2",
        "name": "White Linen Shirt",
        "description": "Lightweight white shirt, business casual.",
        "category": "tops",
        "style": "business casual",
        "season": "Summer",
        "color": "white"
    },
    {
        "id": "3",
        "name": "Black Trousers",
        "description": "Formal black trousers for professional events.",
        "category": "bottoms",
        "style": "professional",
        "season": "Autumn",
        "color": "black"
    },
    {
        "id": "4",
        "name": "Black Leather Loafers",
        "description": "Polished black loafers, ideal for formal settings.",
        "category": "shoes",
        "style": "formal",
        "season": "Autumn",
        "color": "black"
    },
    {
        "id": "5",
        "name": "Silver Stud Earrings",
        "description": "Minimalist silver earrings for understated elegance.",
        "category": "jewelry",
        "style": "formal",
        "season": "All",
        "color": "silver"
    },
    {
        "id": "6",
        "name": "Structured Black Tote",
        "description": "A structured black tote bag suitable for professional use.",
        "category": "bags",
        "style": "professional",
        "season": "Autumn",
        "color": "black"
    },
    # SMART CASUAL
    {
        "id": "7",
        "name": "Beige Cropped Blazer",
        "description": "A light cropped blazer that adds polish to any casual outfit.",
        "category": "outerwear",
        "style": "smart casual",
        "season": "Spring",
        "color": "beige"
    },
    {
        "id": "8",
        "name": "Blue Wide-Leg Jeans",
        "description": "High-waisted wide-leg jeans for casual or smart-casual wear.",
        "category": "bottoms",
        "style": "casual",
        "season": "Spring",
        "color": "blue"
    },
    {
        "id": "9",
        "name": "Striped Cotton Tee",
        "description": "A breathable cotton tee for everyday wear.",
        "category": "tops",
        "style": "casual",
        "season": "Summer",
        "color": "white"
    },
    {
        "id": "10",
        "name": "Canvas Tote Bag",
        "description": "A simple canvas bag for everyday errands.",
        "category": "bags",
        "style": "casual",
        "season": "All",
        "color": "cream"
    },
    {
        "id": "11",
        "name": "White Sneakers",
        "description": "Low-top sneakers great for walking and everyday looks.",
        "category": "shoes",
        "style": "casual",
        "season": "Spring",
        "color": "white"
    },
    # PARTY / COCKTAIL
    {
        "id": "12",
        "name": "Red Satin Slip Dress",
        "description": "A bold red dress perfect for cocktail or evening parties.",
        "category": "dress",
        "style": "cocktail",
        "season": "Summer",
        "color": "red"
    },
    {
        "id": "13",
        "name": "Gold Clutch",
        "description": "A shiny gold clutch for dressy occasions.",
        "category": "bags",
        "style": "night-time party",
        "season": "Summer",
        "color": "gold"
    },
    {
        "id": "14",
        "name": "Strappy Gold Heels",
        "description": "Heels that add elegance to evening looks.",
        "category": "shoes",
        "style": "cocktail",
        "season": "Summer",
        "color": "gold"
    },
    {
        "id": "15",
        "name": "Dangling Crystal Earrings",
        "description": "Sparkly earrings to finish a glam look.",
        "category": "jewelry",
        "style": "night-time party",
        "season": "All",
        "color": "silver"
    }
]


# Embed and upload items to Qdrant
for item in closet_items:
    embedding = model.encode(item['description']).tolist()
    qdrant.upsert(
        collection_name=collection_name,
        points=[PointStruct(
            id=int(item["id"]),
            vector=embedding,
            payload=item
        )]
    )

# --- Step 4: Query vector from user input ---
user_query = "party night out with friends"
query_vector = model.encode(user_query).tolist()

# --- Step 5: Vector search ---
results = qdrant.search(
    collection_name=collection_name,
    query_vector=query_vector,
    limit=5
)

# --- Step 6: Group results by category for LLM prompt ---
# grouped_items = {}
# for hit in results:
#     item = hit.payload
#     cat = item["category"]
#     if cat not in grouped_items:
#         grouped_items[cat] = []
#     grouped_items[cat].append(item["name"])
categories = ["tops", "bottoms", "dress", "shoes", "bags", "jewelry", "outerwear"]

grouped_items = defaultdict(list)

# Loop through each category and get 3 closest items per category
for cat in categories:
    query_filter = Filter(
        must=[
            FieldCondition(
                key="category",
                match=MatchValue(value=cat)
            )
        ]
    )
    
    results = qdrant.search(
        collection_name=collection_name,
        query_vector=query_vector,
        limit=5,
        query_filter=query_filter
    )
    
    for hit in results[:3]:
        item = hit.payload
        entry = f"{item['name']} (ID:{hit.id}): {item['description']}"
        grouped_items[cat].append(entry)

# --- Step 7: Output ---
print(f"🔍 Closest Items for Query: '{user_query}'")
for category, items in grouped_items.items():
    print(f"\n🧺 {category.upper()}:")
    for item in items:
        print("  -", item)


/Users/phonavitra/Desktop/Summer Project 2025/Project 1 AIFashionCompanionApp/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/8d/gp5c13l962z3c391pmyp2bkm0000gn/T/ipykernel_10112/3011693900.py:17: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant.recreate_collection(


🔍 Closest Items for Query: 'party night out with friends'

🧺 TOPS:
  - White Linen Shirt (ID:2): Lightweight white shirt, business casual.
  - Striped Cotton Tee (ID:9): A breathable cotton tee for everyday wear.

🧺 BOTTOMS:
  - Black Trousers (ID:3): Formal black trousers for professional events.
  - Blue Wide-Leg Jeans (ID:8): High-waisted wide-leg jeans for casual or smart-casual wear.

🧺 DRESS:
  - Red Satin Slip Dress (ID:12): A bold red dress perfect for cocktail or evening parties.
  - Black Wrap Dress (ID:1): A black wrap dress suitable for formal events.

🧺 SHOES:
  - Strappy Gold Heels (ID:14): Heels that add elegance to evening looks.
  - Black Leather Loafers (ID:4): Polished black loafers, ideal for formal settings.
  - White Sneakers (ID:11): Low-top sneakers great for walking and everyday looks.

🧺 BAGS:
  - Canvas Tote Bag (ID:10): A simple canvas bag for everyday errands.
  - Structured Black Tote (ID:6): A structured black tote bag suitable for professional use.
  - G

/var/folders/8d/gp5c13l962z3c391pmyp2bkm0000gn/T/ipykernel_10112/3011693900.py:182: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = qdrant.search(
/var/folders/8d/gp5c13l962z3c391pmyp2bkm0000gn/T/ipykernel_10112/3011693900.py:211: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = qdrant.search(


In [3]:
import json
import ollama

# User input"
user_intent = "party night out with friends"
weather = "18°C, cloudy"

# Reuse grouped_items from Qdrant results
# Example: grouped_items = {'dress': [...], 'shoes': [...], ...}

# Convert grouped items to text for prompt
item_text = ""
for category, items in grouped_items.items():
    item_text += f"\n{category.upper()}:\n"
    for item in items:
        item_text += f"  - {item}\n"

# Final prompt to send to LLM
prompt = f"""
You are a fashion stylist. Based on the user’s closet and request, create 2 complete outfits that follow these structure rules:

1. Template A: TOP + BOTTOM + SHOES + BAG + JEWELRY [+OUTERWEAR]
2. Template B: DRESS/OVERALL + SHOES + BAG + JEWELRY [+OUTERWEAR]

The occasion is: {user_intent}  
Weather is: {weather}

Choose items from the closet below. Use each item only once. Use only items that match the occasion and weather. If an item is not suitable, skip it.

Closet Items:
{item_text}

Return your response in the following JSON format:
[
  {{
    "look_name": "...",
    "template": "Template A or B",
    "description": "...",
    "items": [
      {{
        "name": "item name 1",
        "id": "item_id_1",
        "description": "item description 1"
      }},
      {{
        "name": "item name 2",
        "id": "item_id_2",
        "description": "item description 2"
      }}
    ]
  }}
]
"""

response = ollama.chat(
    model='gemma3n:latest',  # or 'mistral'
    messages=[
        {"role": "system", "content": "You are a fashion stylist."},
        {"role": "user", "content": prompt}
    ]
)

print(response['message']['content'])

```json
[
  {
    "look_name": "Glam Party Night",
    "template": "Template A",
    "description": "A chic and sophisticated party look. The red satin slip dress is the focal point, elevated with gold heels and a clutch. The cropped blazer adds a touch of polish for a night out with friends. Dangling crystal earrings complete the glamorous vibe.",
    "items": [
      {
        "name": "Red Satin Slip Dress",
        "id": "12",
        "description": "A bold red dress perfect for cocktail or evening parties."
      },
      {
        "name": "Strappy Gold Heels",
        "id": "14",
        "description": "Heels that add elegance to evening looks."
      },
      {
        "name": "Gold Clutch",
        "id": "13",
        "description": "A shiny gold clutch for dressy occasions."
      },
      {
        "name": "Dangling Crystal Earrings",
        "id": "15",
        "description": "Sparkly earrings to finish a glam look."
      },
      {
        "name": "Beige Cropped Blazer",
  

## Classification


"""
'brand': 'Gucci',
'name': 'Mock Blazer',
'description': 'A stylish black blazer suitable for business casual wear.',
'category': 'Outerwear',
'color': 'Black',
'style': 'Business Casual',
'season': 'Autumn',
"""

### Making Description

In [4]:
prompt = f"""
Using the following fashion item metadata, write a short product description that includes all the details: name, category, color, style, and season.

- Name: {metadata['name']}
- Category: {metadata['category']}
- Color: {metadata['color']}
- Style: {metadata['style']}
- Season: {metadata['season']}

Ensure that all fields are reflected clearly in the description and use neutral language.

Return only the description text without any additional formatting or JSON structure.
"""

response = ollama.chat(
    model='llama3.2:1b',
    messages=[
        {"role": "system", "content": "You are a fashion product description generator."},
        {"role": "user", "content": prompt.strip()}
    ]
)

print("Generated Description:")
description = response['message']['content'].strip()
print(description)

lines = [line.strip() for line in description.splitlines() if line.strip()]

# Get the last non-empty line
clean_description = lines[-1]

print("Cleaned Description:")
print(clean_description)


NameError: name 'metadata' is not defined

### Exposed API from remote server

- **Get local IP network**

    _bash_
    ```  
    ifconfig | grep inet 
    ```

- **Host Ollama in Linux/docker.. right now I am js hosting on MacOS**
    
    _bash_
    ```
    OLLAMA_HOST=0.0.0.0 ollama serve
    ```


In [45]:
import requests

response = requests.post(
    "http://192.168.0.116:11434/api/chat",
    json={
        "model": "gemma3n",
        "messages": [{"role": "user", "content": "Hello from LAN!"}]
    },
    stream=True
)

full_reply = ""
for line in response.iter_lines():
    if line:
        chunk = line.decode("utf-8")
        try:
            content = json.loads(chunk).get("message", {}).get("content", "")
            full_reply += content
        except Exception:
            pass

print("Assistant:", full_reply.strip())


Assistant: Hello from the digital world! 👋 

It's great to hear from you from LAN!  Is there anything I can help you with today?  Let me know what's on your mind. 😊 

(Just a little friendly acknowledgement of your location!)


## Similarity Search From Image

https://docs.autodistill.com/utilities/use-embeddings-in-classification/#embeddingontologyraw-example